<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🛒 Reglas de asociación (Apriori): qué productos electrónicos se compran juntos

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Descubrir qué productos de una tienda de electrónicos se compran juntos para sugerir ventas cruzadas. |
| 🔍 Comprensión de los datos | Cargamos 9 835 transacciones (una por fila, sin encabezado) con hasta 32 productos cada una. |
| 🧹 Preparación de los datos | Convertimos cada fila en una lista de productos y la codificamos como matriz verdadero/falso. |
| 🤖 Modelado | Buscamos conjuntos frecuentes con Apriori (soporte ≥ 1 %) y generamos reglas. |
| 📊 Evaluación | Ordenamos las reglas por confianza y lift y filtramos las accionables. |
| 🚀 Despliegue y conclusiones | Comparamos con FP-Growth y extraemos conclusiones de negocio. |

> 🎯 **Analogía:** un vendedor veterano que, tras mirar miles de tickets, nota que "quien se lleva una laptop y un monitor ViewSonic suele llevarse también un segundo equipo" y acomoda la tienda en consecuencia.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('ElectronidexTransactions2017.csv', 'dia_5')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

# Reglas de asociación (Apriori): análisis de la canasta de compra

El análisis de reglas de asociación es una técnica clave de los minoristas para descubrir qué artículos se compran juntos con frecuencia [4]. En este cuaderno se aplica a las transacciones de una tienda de electrónicos.

### Autores
* Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Descubrir qué productos de una tienda de electrónicos se compran juntos para sugerir ventas cruzadas.

📚 Lectura recomendada: [3]

## Análisis del problema

Las reglas de asociación permiten identificar reglas *fuertes* en datos de transacciones usando medidas de interés (soporte, confianza y lift) [4]. **Pregunta de negocio:** ¿qué productos de la tienda se compran juntos con tanta frecuencia que vale la pena ofrecerlos en paquete o colocarlos cerca?

### Análisis de la canasta de mercado

El análisis de canasta (*market basket analysis*) busca entender el comportamiento de compra: qué artículos coexisten en una misma transacción. Sus resultados apoyan decisiones de ubicación de productos, promociones y recomendaciones [9].

### 🧪 Teoría rápida: reglas de asociación

Una **regla de asociación** tiene la forma **A → B** ("quien compra A suele comprar también B") y se descubre en datos de **transacciones** (canastas de compra) [4]. Se evalúa con tres medidas:

| Medida | Fórmula | Se lee como |
|---|---|---|
| **Soporte** de A | $\text{sop}(A)=\dfrac{\#\{\text{transacciones que contienen }A\}}{\#\{\text{transacciones}\}}$ | qué tan frecuente es el conjunto |
| **Confianza** de A → B | $\text{conf}(A\to B)=\dfrac{\text{sop}(A\cup B)}{\text{sop}(A)}$ | probabilidad de B dado A |
| **Elevación** (*lift*) | $\text{lift}(A\to B)=\dfrac{\text{conf}(A\to B)}{\text{sop}(B)}$ | cuántas veces más probable es B si hay A, frente a B sin saber nada |

Un *lift* > 1 indica asociación positiva, = 1 independencia y < 1 que se **sustituyen** [6]. Otras medidas útiles son el **apalancamiento** (*leverage*) $\text{sop}(A\cup B)-\text{sop}(A)\,\text{sop}(B)$ y la **convicción** $\dfrac{1-\text{sop}(B)}{1-\text{conf}(A\to B)}$ [10].

**Algoritmo Apriori.** Se apoya en una propiedad: *todo subconjunto de un conjunto frecuente también es frecuente* (y, al revés, si un conjunto no es frecuente, ninguno de sus superconjuntos lo es), lo que permite **podar** candidatos [5]. El proceso tiene dos pasos: (1) hallar los **conjuntos frecuentes** con soporte ≥ `min_support`; (2) generar las **reglas** con confianza (o lift) ≥ umbral [9]. Alternativas más rápidas son **FP-Growth** [7] y **Eclat** [8].

In [ ]:
%pip install -q squarify mlxtend

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
sns.set_style('whitegrid')

# Reglas de asociación (mlxtend)
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos 9 835 transacciones (una por fila, sin encabezado) con hasta 32 productos cada una.

📚 Lectura recomendada: [3]

In [ ]:
# El archivo NO tiene encabezado: cada fila es una transacción (la primera fila es una compra más)
data = pd.read_csv('sample_data/ElectronidexTransactions2017.csv', header=None)
print('Transacciones y columnas:', data.shape)
data.head()

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos cada fila en una lista de productos y la codificamos como matriz verdadero/falso.

📚 Lectura recomendada: [4] [6]

### Preparación de los datos

Cada fila es una transacción y las columnas son posiciones, no productos; las celdas vacías (`NaN`) solo rellenan. El `TransactionEncoder` de `mlxtend` [11] necesita **listas de productos** y entrega una matriz verdadero/falso con una columna por producto. Primero se construye cada lista descartando las celdas vacías.

In [ ]:
transacciones = [[p for p in fila if isinstance(p, str)] for fila in data.values.tolist()]
print('Transacciones:', len(transacciones))
print('Productos por transacción: promedio =', round(sum(map(len, transacciones)) / len(transacciones), 2), '| máximo =', max(map(len, transacciones)))
print('Transacciones vacías:', sum(len(t) == 0 for t in transacciones))

In [ ]:
t = TransactionEncoder()
data = pd.DataFrame(t.fit_transform(transacciones), columns=t.columns_)
print('Matriz transacciones × productos:', data.shape)

In [ ]:
print('¿Hay una columna "nan"?', 'nan' in data.columns)

### Visualización de los productos más frecuentes

In [ ]:
r = data.sum(axis=0).sort_values(ascending=False)[:20]
plt.figure(figsize=(12, 5))
sns.barplot(x=r.index, y=r.values, color='steelblue')
plt.xticks(rotation=90)
plt.ylabel('Número de transacciones')
plt.title('20 productos más comprados')
plt.tight_layout()
plt.show()

In [ ]:
cmap = matplotlib.cm.Blues
norm = matplotlib.colors.Normalize(vmin=r.values.min(), vmax=r.values.max())
colors = [cmap(norm(v)) for v in r.values]
plt.figure(figsize=(10, 7))
squarify.plot(sizes=r.values, label=r.index, alpha=.7, color=colors, text_kwargs={'fontsize': 8})
plt.title('Mapa de árbol de los 20 productos más comprados')
plt.axis('off')
plt.show()

---
## 🤖 Fase 4 · Modelado

Buscamos conjuntos frecuentes con Apriori (soporte ≥ 1 %) y generamos reglas.

📚 Lectura recomendada: [5] [11]

### Algoritmo Apriori

Buscamos los conjuntos de productos que aparecen en **al menos el 1 %** de las transacciones (`min_support = 0.01`).

In [ ]:
freq_items = apriori(data, min_support=0.01, use_colnames=True)
freq_items['longitud'] = freq_items['itemsets'].apply(len)
print('Conjuntos frecuentes:', len(freq_items))
print(freq_items['longitud'].value_counts().sort_index().rename('conjuntos').to_frame())

In [ ]:
freq_items.sort_values('support', ascending=False).head(10)

Con soporte mínimo de 1 %, el primer paso de Apriori entrega los conjuntos frecuentes (arriba se cuentan por tamaño). Los productos individuales más frecuentes dominan la lista; los conjuntos de 2 y 3 productos son los candidatos a reglas.

In [ ]:
# Reglas con confianza >= 30 % y asociación positiva (lift > 1)
res = association_rules(freq_items, metric='confidence', min_threshold=0.3)
res = res[res['lift'] > 1].sort_values('lift', ascending=False)
print('Reglas generadas:', len(res))

In [ ]:
res[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

### Interpretación
- **Soporte:** fracción de transacciones que contienen *todos* los productos de la regla.
- **Confianza:** de las transacciones con el antecedente, qué fracción incluye también el consecuente.
- **Lift:** cuántas veces más probable es el consecuente cuando aparece el antecedente que en general.

Las reglas con mayor lift giran en torno a computadoras y monitores (por ejemplo, **HP Laptop + Acer Aspire → ViewSonic Monitor**, lift ≈ 3.4). Son hallazgos de *asociación*, no de causalidad: que dos productos se compren juntos no significa que uno cause la compra del otro [10].

---
## 📊 Fase 5 · Evaluación

Ordenamos las reglas por confianza y lift y filtramos las accionables.

📚 Lectura recomendada: [10]

### Seleccionar y filtrar los resultados

In [ ]:
frequent_itemsets = apriori(data, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.sort_values('support', ascending=False)

In [ ]:
association_rules(frequent_itemsets, metric='lift', min_threshold=1)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

In [ ]:
# Pares de productos (length = 2) con soporte >= 1 %
frequent_itemsets[(frequent_itemsets['length'] == 2) & (frequent_itemsets['support'] >= 0.01)]

In [ ]:
# Productos individuales (length = 1) con soporte >= 1 %
frequent_itemsets[(frequent_itemsets['length'] == 1) & (frequent_itemsets['support'] >= 0.01)]

---
## 🧪 Refuerzo: soporte, confianza y lift calculados a mano

Antes de confiar en la librería, conviene entender de dónde salen los números. Con una canasta mínima de 5 compras calculamos las medidas con conteos y comprobamos que `mlxtend` [11] da lo mismo.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

mini = [['pan', 'leche'], ['pan', 'leche', 'huevos'], ['pan', 'huevos'], ['leche', 'huevos'], ['pan', 'leche', 'cafe']]
n = len(mini)
sop = lambda s: sum(set(s) <= set(t) for t in mini) / n

print('sop(pan)         =', sop(['pan']))
print('sop(pan, leche)  =', sop(['pan', 'leche']))
print('conf(pan→leche)  =', sop(['pan', 'leche']) / sop(['pan']))
print('lift(pan→leche)  =', sop(['pan', 'leche']) / (sop(['pan']) * sop(['leche'])))

te = TransactionEncoder()
M = pd.DataFrame(te.fit_transform(mini), columns=te.columns_)
reglas = association_rules(apriori(M, min_support=0.4, use_colnames=True), metric='confidence', min_threshold=0.5)
reglas[(reglas.antecedents == frozenset({'pan'})) & (reglas.consequents == frozenset({'leche'}))][['support', 'confidence', 'lift']]

### ✍️ Ejercicios
1. Calcule a mano la confianza y el lift de `huevos → leche` y verifique con `mlxtend`.
2. ¿Puede una regla tener confianza alta y lift menor que 1? Construya un ejemplo.
3. Si baja `min_support`, ¿qué pasa con la cantidad de conjuntos frecuentes y con el tiempo de cómputo? [5]

---
## 🧪 Refuerzo: FP-Growth frente a Apriori: ¿mismos resultados, distinto tiempo?

**FP-Growth** construye un árbol compacto de patrones frecuentes y evita generar candidatos [7]; por eso suele escalar mejor que Apriori cuando hay muchas transacciones o un soporte mínimo muy bajo. Lo que sí debe cumplirse siempre es que ambos algoritmos devuelvan **los mismos conjuntos frecuentes**. Lo comprobamos y medimos el tiempo con la matriz de Electronidex.

In [ ]:
import time
from mlxtend.frequent_patterns import fpgrowth

print('Soporte | conjuntos frecuentes | mismos resultados | Apriori (ms) | FP-Growth (ms)')
for s in [0.01, 0.02, 0.05]:
    t0 = time.perf_counter(); a = apriori(data, min_support=s, use_colnames=True); ta = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); b = fpgrowth(data, min_support=s, use_colnames=True); tb = (time.perf_counter() - t0) * 1000
    print(f'{s:7.2f} | {len(a):20d} | {str(set(a.itemsets) == set(b.itemsets)):17s} | {ta:12.1f} | {tb:14.1f}')

### ✍️ Ejercicios
1. Repita con `min_support=0.005`. ¿Cambia cuál algoritmo es más rápido? ¿Por qué la ventaja de FP-Growth depende del tamaño de los datos?
2. Filtre las reglas con `lift > 2` y `confidence > 0.5`. ¿Cuáles son *accionables* (por ejemplo, para ofrecer un monitor junto con una laptop)?
3. Una regla con lift alto pero soporte muy bajo, ¿es confiable? Discuta con `leverage` y `conviction` [10].

---
## ✅ Conclusiones y siguientes pasos

Con 9 835 transacciones y 131 productos, Apriori (soporte ≥ 1 %) halló 370 conjuntos frecuentes y 153 reglas con confianza ≥ 30 % y lift > 1. Las asociaciones más fuertes combinan computadoras y monitores (lift hasta ≈ 3.4), útiles para paquetes o ventas cruzadas, aunque son asociaciones y no causas [10]. Además, el CSV no tiene encabezado: leerlo con `header=0` pierde la primera compra y distorsiona el conteo. FP-Growth devolvió los mismos conjuntos, pero en este conjunto pequeño no fue más rápido: su ventaja aparece con más datos o soportes más bajos [7].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [12], [13], [14].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Jiménez Delgado, E. (s. f.). *fiec_intro_machine_learning: ElectronidexTransactions2017.csv* [Transacciones de una tienda de electrónica]. GitLab. https://gitlab.com/ejimenez/fiec_intro_machine_learning

**[4]** Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data*, 207–216. https://doi.org/10.1145/170035.170072

**[5]** Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proceedings of the 20th International Conference on Very Large Data Bases (VLDB)*, 487–499.

**[6]** Brin, S., Motwani, R., Ullman, J. D., & Tsur, S. (1997). Dynamic itemset counting and implication rules for market basket data. *Proceedings of the 1997 ACM SIGMOD International Conference on Management of Data*, 255–264.

**[7]** Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proceedings of the 2000 ACM SIGMOD International Conference on Management of Data*, 1–12.

**[8]** Zaki, M. J. (2000). Scalable algorithms for association mining. *IEEE Transactions on Knowledge and Data Engineering, 12*(3), 372–390. https://doi.org/10.1109/69.846291

**[9]** Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3.ª ed.). Morgan Kaufmann. https://doi.org/10.1016/C2009-0-61819-5

**[10]** Tan, P.-N., Steinbach, M., Karpatne, A., & Kumar, V. (2019). *Introduction to Data Mining* (2.ª ed.). Pearson.

**[11]** Raschka, S. (2018). MLxtend: Providing machine learning and data science utilities and extensions to Python's scientific computing stack. *Journal of Open Source Software, 3*(24), 638. https://doi.org/10.21105/joss.00638

**[12]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[13]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**[14]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55